In [7]:
import yaml
import numpy as np
import polars as pl

patho_labels  = ['Pathogenic', 'Likely_pathogenic']
benign_labels = ['Benign', 'Likely_benign']

clinvar_labels = patho_labels + benign_labels

# Create input for VEP and annotation pipeline

In [ ]:
# CLINVAR_URL = "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/clinvar_20260621.vcf.gz"

# !wget -P /s/project/ukbbgym/annotation_files/clinvar/ {CLINVAR_URL}


In [ ]:
annotation_dir = "/s/project/ukbbgym/annotation_files/clinvar"
clinvar_vcf_path = "/s/project/ukbbgym/annotation_files/clinvar/clinvar_20260621.vcf.gz"

clinvar = (
    pl.scan_csv(
        clinvar_vcf_path,
        separator="\t",
        comment_prefix="##",
        schema_overrides={"#CHROM": pl.Utf8, "POS": pl.Int64},
        ignore_errors=True,
    )
    .rename({"#CHROM": "chrom", "POS": "pos", "REF": "ref", "ALT": "alt"})
    .with_columns(
        chrom="chr" + pl.col("chrom").cast(pl.Utf8).str.replace(r"^chr", ""),
        clinical_significance=pl.col("INFO").str.extract(r"CLNSIG=([^;]+)", 1),
    )
    .with_columns(
        id=pl.concat_str(["chrom", "pos", "ref", "alt"], separator=":"),
    )
    .select(["id", "chrom", "pos", "ref", "alt", "clinical_significance"])
    .filter(pl.col("clinical_significance").is_in(clinvar_labels))
    .unique(subset="id")          # one row per variant
    .collect()
)

# 1. variant_metadata.parquet — exactly the schema the Snakefile consumes
clinvar.select("id", "chrom", "pos", "ref", "alt").write_parquet(
    f"{annotation_dir}/variant_metadata.parquet"
)

# 2. keep the labels to join back onto the final annotation output on `id`
clinvar.select("id", "clinical_significance").write_parquet(
    f"{annotation_dir}/clinvar_labels.parquet"
)


# Add more missense annotations

In [8]:
import add_more_annotations as ann

In [9]:
cv_all = (
    pl.read_parquet("/s/project/ukbbgym/annotation_files/clinvar/vep_annotations_processed_cadd_fill_na.parquet")
)

cv_all

id,chrom,pos,ref,alt,region,cds_position,protein_position,distance,amino_acids,gnomade_af,gnomadg_af,polyphen,sift,cadd_phred,cadd_raw,am_pathogenicity,loftee_hc,loftee_lc,relative_cds_position,next_in_frame_relative,spliceai_delta_score,pangolin_score,delta_score,absplice_dna_max,absplice2_max,five_prime_utr_variant_consequence_uaug_gained,five_prime_utr_variant_consequence_uaug_lost,five_prime_utr_variant_consequence_uframeshift,five_prime_utr_variant_consequence_ustop_gained,five_prime_utr_variant_consequence_ustop_lost,variant_length,gpn_score,promoterai,score_pai3d,blosum62,is_indel,…,gc_is_na,gerpn_is_na,gerps_is_na,grantham_is_na,remapoverlapcl_is_na,remapoverlaptf_is_na,roulette-ar_is_na,roulette-mr_is_na,zoopriphylop_is_na,zoouce_is_na,zooverphylop_is_na,bstatistic_is_na,cdnapos_is_na,dbscsnv-ada_score_is_na,dbscsnv-rf_score_is_na,mamphcons_is_na,mamphylop_is_na,mindisttse_is_na,mindisttss_is_na,mirsvr-aln_is_na,mirsvr-e_is_na,mirsvr-score_is_na,motifdist_is_na,motifecount_is_na,motifehipos_is_na,motifescorechng_is_na,priphcons_is_na,priphylop_is_na,relcdspos_is_na,relprotpos_is_na,relcdnapos_is_na,toverlapmotifs_is_na,targetscan_is_na,verphcons_is_na,verphylop_is_na,cpt1_llr_is_na,blosum62_is_na
str,str,i64,str,str,str,str,str,i64,str,f32,f32,f32,f32,f32,f32,f32,i8,i8,f32,f32,f32,f32,f32,f32,f32,u8,u8,u8,u8,u8,u32,f32,f32,f32,f32,i8,…,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8
"""chr4:102689649:A:G""","""chr4""",102689649,"""A""","""G""","""ENSG00000109323""","""885/2640""","""295/879""",null,"""H""",6.8570e-7,0.000007,0.0,1.0,6.106,0.56391,0.0,0,0,0.34,0.0,0.02,0.01,0.02,0.003,0.000332,0,0,0,0,0,1,-2.84,0.0,0.0,0.0,0,…,0,0,0,1,1,1,1,0,0,1,0,0,0,1,1,0,0,0,0,1,1,1,1,1,1,1,0,0,0,0,0,1,1,0,0,1,1
"""chr4:102689661:C:T""","""chr4""",102689661,"""C""","""T""","""ENSG00000109323""","""873/2640""","""291/879""",null,"""W/*""",0.000001,null,0.0,1.0,39.0,8.936202,0.0,1,0,0.33,0.0,0.01,0.07,0.0,0.003,0.00202,0,0,0,0,0,1,-9.01,0.0,0.0,0.0,0,…,0,0,0,1,1,1,1,0,0,1,0,0,0,1,1,0,0,0,0,1,1,1,1,1,1,1,0,0,0,0,0,1,1,0,0,1,1
"""chr4:102690583:T:G""","""chr4""",102690583,"""T""","""G""","""ENSG00000109323""",null,null,null,null,0.000573,0.000263,0.0,1.0,9.554,0.929807,0.0,0,0,0.0,0.0,0.01,0.0,0.0,0.003,0.000315,0,0,0,0,0,1,-2.43,0.0,0.0,0.0,0,…,0,0,0,1,1,1,1,0,0,1,0,0,1,1,1,0,0,0,0,1,1,1,1,1,1,1,0,0,1,1,1,1,1,0,0,1,1
"""chr4:102690671:C:T""","""chr4""",102690671,"""C""","""T""","""ENSG00000109323""","""774/2640""","""258/879""",null,"""L""",0.000056,0.000007,0.0,1.0,6.819,0.63858,0.0,0,0,0.29,0.0,0.0,0.0,0.0,0.003,0.000296,0,0,0,0,0,1,-2.44,0.0,0.0,0.0,0,…,0,0,0,1,0,0,1,0,0,1,0,0,0,1,1,0,0,0,0,1,1,1,1,1,1,1,0,0,0,0,0,1,1,0,0,1,1
"""chr4:102690674:C:T""","""chr4""",102690674,"""C""","""T""","""ENSG00000109323""","""771/2640""","""257/879""",null,"""K""",0.000001,null,0.0,1.0,2.954,0.270028,0.0,0,0,0.29,0.0,0.0,0.0,0.0,0.004,0.000296,0,0,0,0,0,1,3.0,0.0,0.0,0.0,0,…,0,0,0,1,0,0,1,0,0,1,0,0,0,1,1,0,0,0,0,1,1,1,1,1,1,1,0,0,0,0,0,1,1,0,0,1,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr5:140647688:GC:G""","""chr5""",140647688,"""GC""","""G""","""ENSG00000113119""",null,null,2285,null,0.000668,0.006938,0.0,1.0,15.81,1.803421,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,2,-1.41,0.0,0.0,0.0,1,…,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
"""chr2:43912558:CCT:C""","""chr2""",43912558,"""CCT""","""C""","""ENSG00000138095""",null,null,null,null,null,null,0.0,1.0,0.0,0.28051,0.0,1,0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0,0,0,0,0,3,-1.41,0.0,0.0,0.0,1,…,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
"""chr11:125602442:G:GT""","""chr11""",125602442,"""G""","""GT""","""ENSG00000134910""",null,null,null,null,0.03097,0.001114,0.0,1.0,0.601,-0.146952,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,2,-1.41,0.0,0.0,0.0,1,…,1,1,1,1,1,1,1,1

In [10]:
ann.main(
    "/s/project/ukbbgym/annotation_files/clinvar/vep_annotations_processed_cadd_fill_na.parquet",
    fill_null_defaults_path="fill_null_defaults.yaml",
    output_path="/s/project/ukbbgym/annotation_files/clinvar/vep_annotations_processed_cadd_fill_na_more.parquet",
    download_dir="/s/project/ukbbgym/annotation_files/more_annotations",
    add_alphamissense=False,
    add_cpt1=False,
    add_cadd=False,
    add_gpn_msa=False,
    add_phylop=False,
    add_next_in_frame=False,
)

[2026-06-24 21:06:57,271] INFO:add_more_annotations: === Downloading required files ===
[2026-06-24 21:06:57,274] INFO:add_more_annotations:   Already complete: /s/project/ukbbgym/annotation_files/more_annotations/tmp/clinvar.vcf.gz
[2026-06-24 21:06:57,275] INFO:add_more_annotations: === Auto-downloading annotation scores ===
[2026-06-24 21:06:57,281] INFO:add_more_annotations:   Already complete: /s/project/ukbbgym/annotation_files/more_annotations/tmp/revel.zip
[2026-06-24 21:06:57,288] INFO:add_more_annotations:   Already extracted: /s/project/ukbbgym/annotation_files/more_annotations/tmp/revel.zip -> /s/project/ukbbgym/annotation_files/more_annotations/tmp/revel
[2026-06-24 21:06:57,291] INFO:add_more_annotations:   Already present: /s/project/ukbbgym/annotation_files/more_annotations/tmp/ClinPred_hg38.txt.gz
[2026-06-24 21:06:57,292] INFO:add_more_annotations:   Already present: /s/project/ukbbgym/annotation_files/more_annotations/tmp/bayesdel.gz
[2026-06-24 21:06:57,296] INFO:ad

'/s/project/ukbbgym/annotation_files/clinvar/vep_annotations_processed_cadd_fill_na_more.parquet'

# Add clinical significance labels to the final annotation output

In [11]:
cv_all = (
    pl.read_parquet("/s/project/ukbbgym/annotation_files/clinvar/vep_annotations_processed_cadd_fill_na_more.parquet")
)

cv_all

id,chrom,pos,ref,alt,region,cds_position,protein_position,distance,amino_acids,gnomade_af,gnomadg_af,polyphen,sift,cadd_phred,cadd_raw,am_pathogenicity,loftee_hc,loftee_lc,relative_cds_position,next_in_frame_relative,spliceai_delta_score,pangolin_score,delta_score,absplice_dna_max,absplice2_max,five_prime_utr_variant_consequence_uaug_gained,five_prime_utr_variant_consequence_uaug_lost,five_prime_utr_variant_consequence_uframeshift,five_prime_utr_variant_consequence_ustop_gained,five_prime_utr_variant_consequence_ustop_lost,variant_length,gpn_score,promoterai,score_pai3d,blosum62,is_indel,…,core_promoter,proximal_promoter,encode_enhancer,encode_promoter,1bp_del,1bp_ins,2_5bp_del,2_5bp_ins,gt_5bp_del,gt_5bp_ins,is_indel_is_na,is_insertion_is_na,is_deletion_is_na,encode_pls_is_na,encode_pels_is_na,encode_dels_is_na,encode_ca_is_na,encode_tf_is_na,not_annotated_in_encode_is_na,revel_score_is_na,clinpred_score_is_na,bayes_del_is_na,popeve_is_na,mobi_full_disorder_priority_is_na,mobi_curated_disorder_priority_is_na,mobi_full_lip_priority_is_na,ted_domain_is_na,low_complexity_domain_is_na,loftee_hc_is_na,loftee_lc_is_na,plddt,is_pioneer_interface_high,clinical_significance,clinvar_patho,clinvar_likely_patho,clinvar_benign,clinvar_likely_benign
str,str,i64,str,str,str,str,str,i64,str,f32,f32,f32,f32,f32,f32,f32,i8,i8,f32,f32,f32,f32,f32,f32,f32,u8,u8,u8,u8,u8,u32,f32,f32,f32,f32,i8,…,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,f32,i8,str,i8,i8,i8,i8
"""chr4:102689649:A:G""","""chr4""",102689649,"""A""","""G""","""ENSG00000109323""","""885/2640""","""295/879""",null,"""H""",6.8570e-7,0.000007,0.0,1.0,6.106,0.56391,0.0,0,0,0.34,0.0,0.02,0.01,0.02,0.003,0.000332,0,0,0,0,0,1,-2.84,0.0,0.0,0.0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,0,0,0,0,0,0,0,98.309998,0,"""Likely_benign""",0,0,0,1
"""chr4:102689661:C:T""","""chr4""",102689661,"""C""","""T""","""ENSG00000109323""","""873/2640""","""291/879""",null,"""W/*""",0.000001,null,0.0,1.0,39.0,8.936202,0.0,1,0,0.33,0.0,0.01,0.07,0.0,0.003,0.00202,0,0,0,0,0,1,-9.01,0.0,0.0,0.0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,0,0,0,0,0,98.879997,0,"""Pathogenic""",1,0,0,0
"""chr4:102690583:T:G""","""chr4""",102690583,"""T""","""G""","""ENSG00000109323""",null,null,null,null,0.000573,0.000263,0.0,1.0,9.554,0.929807,0.0,0,0,0.0,0.0,0.01,0.0,0.0,0.003,0.000315,0,0,0,0,0,1,-2.43,0.0,0.0,0.0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,0,0,0,0,0,0,0,null,0,"""Benign""",0,0,1,0
"""chr4:102690671:C:T""","""chr4""",102690671,"""C""","""T""","""ENSG00000109323""","""774/2640""","""258/879""",null,"""L""",0.000056,0.000007,0.0,1.0,6.819,0.63858,0.0,0,0,0.29,0.0,0.0,0.0,0.0,0.003,0.000296,0,0,0,0,0,1,-2.44,0.0,0.0,0.0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,0,0,0,0,0,0,0,97.620003,0,"""Likely_benign""",0,0,0,1
"""chr4:102690674:C:T""","""chr4""",102690674,"""C""","""T""","""ENSG00000109323""","""771/2640""","""257/879""",null,"""K""",0.000001,null,0.0,1.0,2.954,0.270028,0.0,0,0,0.29,0.0,0.0,0.0,0.0,0.004,0.000296,0,0,0,0,0,1,3.0,0.0,0.0,0.0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,0,0,0,0,0,0,0,97.059998,0,"""Likely_benign""",0,0,0,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr5:140647688:GC:G""","""chr5""",140647688,"""GC""","""G""","""ENSG00000113119""",null,null,2285,null,0.000668,0.006938,0.0,1.0,15.81,1.803421,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,2,-1.41,0.0,0.0,0.0,1,…,1,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,0,0,0,0,0,0,0,null,0,"""Likely_benign""",0,0,0,1
"""chr2:43912558:CCT:C""","""chr2""",43912558,"""CCT""","""C""","""ENSG00000138095""",null,null,null,null,null,null,0.0,1.0,0.0,0.28051,0.0,1,0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0,0,0,0,0,3,-1.41,0.0,0.0,0.0,1,…,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,0,0,0,0,0,0,0,null,0,"""Likely_pathogenic""",0,1,0,0
"""chr11:125602442:G:GT""","""chr11""",

In [12]:
cv_all.columns

['id',
 'chrom',
 'pos',
 'ref',
 'alt',
 'region',
 'cds_position',
 'protein_position',
 'distance',
 'amino_acids',
 'gnomade_af',
 'gnomadg_af',
 'polyphen',
 'sift',
 'cadd_phred',
 'cadd_raw',
 'am_pathogenicity',
 'loftee_hc',
 'loftee_lc',
 'relative_cds_position',
 'next_in_frame_relative',
 'spliceai_delta_score',
 'pangolin_score',
 'delta_score',
 'absplice_dna_max',
 'absplice2_max',
 'five_prime_utr_variant_consequence_uaug_gained',
 'five_prime_utr_variant_consequence_uaug_lost',
 'five_prime_utr_variant_consequence_uframeshift',
 'five_prime_utr_variant_consequence_ustop_gained',
 'five_prime_utr_variant_consequence_ustop_lost',
 'variant_length',
 'gpn_score',
 'promoterai',
 'score_pai3d',
 'blosum62',
 'is_indel',
 'is_insertion',
 'is_deletion',
 'mobi_full_disorder_priority',
 'mobi_curated_disorder_priority',
 'mobi_full_lip_priority',
 'ted_domain',
 'low_complexity_domain',
 'cpt1_llr',
 'encode_pls',
 'encode_pels',
 'encode_dels',
 'encode_ca',
 'encode_ca-ctc

In [13]:
annotation_dir = "/s/project/ukbbgym/annotation_files/clinvar"
cv_labels = pl.read_parquet(f"{annotation_dir}/clinvar_labels_20260621.parquet")
cv_labels

id,clinical_significance
str,str
"""chr6:161360200:G:A""","""Likely_benign"""
"""chr21:44287497:C:T""","""Likely_benign"""
"""chr17:67911214:A:G""","""Likely_benign"""
"""chr17:1676275:G:A""","""Likely_benign"""
"""chr11:6640711:A:G""","""Likely_benign"""
…,…
"""chr19:39499413:C:G""","""Likely_benign"""
"""chr19:17787812:C:T""","""Likely_benign"""
"""chr3:184355543:G:C""","""Likely_benign"""


In [14]:
cv_all_labs = (
    cv_labels
    .join(
        cv_all,
        on="id",
        # how="left",
        validate="1:m"
    )
)

cv_all_labs

id,clinical_significance,chrom,pos,ref,alt,region,cds_position,protein_position,distance,amino_acids,gnomade_af,gnomadg_af,polyphen,sift,cadd_phred,cadd_raw,am_pathogenicity,loftee_hc,loftee_lc,relative_cds_position,next_in_frame_relative,spliceai_delta_score,pangolin_score,delta_score,absplice_dna_max,absplice2_max,five_prime_utr_variant_consequence_uaug_gained,five_prime_utr_variant_consequence_uaug_lost,five_prime_utr_variant_consequence_uframeshift,five_prime_utr_variant_consequence_ustop_gained,five_prime_utr_variant_consequence_ustop_lost,variant_length,gpn_score,promoterai,score_pai3d,blosum62,…,core_promoter,proximal_promoter,encode_enhancer,encode_promoter,1bp_del,1bp_ins,2_5bp_del,2_5bp_ins,gt_5bp_del,gt_5bp_ins,is_indel_is_na,is_insertion_is_na,is_deletion_is_na,encode_pls_is_na,encode_pels_is_na,encode_dels_is_na,encode_ca_is_na,encode_tf_is_na,not_annotated_in_encode_is_na,revel_score_is_na,clinpred_score_is_na,bayes_del_is_na,popeve_is_na,mobi_full_disorder_priority_is_na,mobi_curated_disorder_priority_is_na,mobi_full_lip_priority_is_na,ted_domain_is_na,low_complexity_domain_is_na,loftee_hc_is_na,loftee_lc_is_na,plddt,is_pioneer_interface_high,clinical_significance_right,clinvar_patho,clinvar_likely_patho,clinvar_benign,clinvar_likely_benign
str,str,str,i64,str,str,str,str,str,i64,str,f32,f32,f32,f32,f32,f32,f32,i8,i8,f32,f32,f32,f32,f32,f32,f32,u8,u8,u8,u8,u8,u32,f32,f32,f32,f32,…,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,f32,i8,str,i8,i8,i8,i8
"""chr4:102689649:A:G""","""Likely_benign""","""chr4""",102689649,"""A""","""G""","""ENSG00000109323""","""885/2640""","""295/879""",null,"""H""",6.8570e-7,0.000007,0.0,1.0,6.106,0.56391,0.0,0,0,0.34,0.0,0.02,0.01,0.02,0.003,0.000332,0,0,0,0,0,1,-2.84,0.0,0.0,0.0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,0,0,0,0,0,0,0,98.309998,0,"""Likely_benign""",0,0,0,1
"""chr4:102689661:C:T""","""Pathogenic""","""chr4""",102689661,"""C""","""T""","""ENSG00000109323""","""873/2640""","""291/879""",null,"""W/*""",0.000001,null,0.0,1.0,39.0,8.936202,0.0,1,0,0.33,0.0,0.01,0.07,0.0,0.003,0.00202,0,0,0,0,0,1,-9.01,0.0,0.0,0.0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,0,0,0,0,0,98.879997,0,"""Pathogenic""",1,0,0,0
"""chr4:102690583:T:G""","""Benign""","""chr4""",102690583,"""T""","""G""","""ENSG00000109323""",null,null,null,null,0.000573,0.000263,0.0,1.0,9.554,0.929807,0.0,0,0,0.0,0.0,0.01,0.0,0.0,0.003,0.000315,0,0,0,0,0,1,-2.43,0.0,0.0,0.0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,0,0,0,0,0,0,0,null,0,"""Benign""",0,0,1,0
"""chr4:102690671:C:T""","""Likely_benign""","""chr4""",102690671,"""C""","""T""","""ENSG00000109323""","""774/2640""","""258/879""",null,"""L""",0.000056,0.000007,0.0,1.0,6.819,0.63858,0.0,0,0,0.29,0.0,0.0,0.0,0.0,0.003,0.000296,0,0,0,0,0,1,-2.44,0.0,0.0,0.0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,0,0,0,0,0,0,0,97.620003,0,"""Likely_benign""",0,0,0,1
"""chr4:102690674:C:T""","""Likely_benign""","""chr4""",102690674,"""C""","""T""","""ENSG00000109323""","""771/2640""","""257/879""",null,"""K""",0.000001,null,0.0,1.0,2.954,0.270028,0.0,0,0,0.29,0.0,0.0,0.0,0.0,0.004,0.000296,0,0,0,0,0,1,3.0,0.0,0.0,0.0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,0,0,0,0,0,0,0,97.059998,0,"""Likely_benign""",0,0,0,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr5:140647688:GC:G""","""Likely_benign""","""chr5""",140647688,"""GC""","""G""","""ENSG00000113119""",null,null,2285,null,0.000668,0.006938,0.0,1.0,15.81,1.803421,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,2,-1.41,0.0,0.0,0.0,…,1,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,0,0,0,0,0,0,0,null,0,"""Likely_benign""",0,0,0,1
"""chr2:43912558:CCT:C""","""Likely_pathogenic""","""chr2""",43912558,"""CCT""","""C""","""ENSG00000138095""",null,null,null,null,null,null,0.0,1.0,0.0,0.28051,0.0,1,0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0,0,0,0,0,3,-1.41,0.0,0.0,0.0,

In [15]:
cv_all_labs.write_parquet(
    f"{annotation_dir}/clinvar_significance_vep_annotations_processed_cadd_fill_na_20260621.parquet"
)